### TASK IN HAND:

* PART A: We'll divide our documents into *__chunks__*,
* PART B: We'll encode our *__chunks__* into *__vectors__* and store in *__chroma__*,
* PART C: We'll visualize our *__vectors__*

#### PART A: DIVIDE OUR DOCUMENTS INTO CHUNKS

In [1]:
import os
import glob
import tiktoken
import numpy as np
from rich import print
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

C:\Users\Ujjwal Singh\AppData\Local\Temp\ipykernel_33300\510120170.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


These imports set up a complete __Retrieval-Augmented Generation (RAG)__ pipeline. This code gathers text documents, splits them into manageable chunks, converts those chunks into mathematical vectors (embeddings) using OpenAI or Hugging Face, stores them in a vector database (Chroma), and then runs a statistical analysis (t-SNE) to plot and visualize those vectors in a 3D interactive graph (Plotly).

#### What These Imports Do (Grouped by Function)

1. _File Handling & System Utilities_

* `import os` & `import glob`: Built-in modules used to navigate your operating system folders and search for specific files using wildcards (like finding all `.txt` files).

* `from dotenv import load_dotenv`: Loads private credentials (like API keys) from a hidden `.env` file securely into your script memory.

2. _Core Math & AI Tokenization_

* `import tiktoken`: OpenAI's rapid text tokenizer that counts how many structural chunks ("tokens") your text contains before sending it to an AI model.

* `import numpy as np`: The foundational math library for handling large multidimensional grids of numbers (arrays and matrices).

3. _The LangChain Framework (Data Ingestion & Vector Storage)_

* `from langchain_community.document_loaders import DirectoryLoader, TextLoader`: Automates opening an entire folder directory and reading the text contents inside it.

* `from langchain_text_splitters import RecursiveCharacterTextSplitter`: Smartly chops long documents into smaller, overlapping paragraphs so they don't exceed AI context windows.

* `from langchain_openai import OpenAIEmbeddings` & `from langchain_huggingface import HuggingFaceEmbeddings`: Generates high-dimensional numeric vectors (embeddings) from your text using either OpenAI’s cloud API or Hugging Face's open-source models.

* `from langchain_chroma import Chroma`: A lightweight, local vector database used to store and quickly search through your document embeddings.

4. _Data Visualization & Analytics_

* `from sklearn.manifold import TSNE`: An advanced machine learning algorithm (t-Distributed Stochastic Neighbor Embedding). It takes your complex, high-dimensional AI vectors (which can have over 1,500 dimensions) and squashes them down to 2D or 3D coordinates so human eyes can see how texts cluster together.

* `import plotly.graph_objects` as `go`: An interactive graphing library. It takes those 3D coordinates from t-SNE and displays them as a rotatable, clickable scatter plot graph in your browser or notebook.

In [2]:
# Loading environment variables from .env file
db_name = "vector_db"

load_dotenv(override=True)

groq_api_key = os.getenv("GROQ_API_KEY")
groq_model = os.getenv("GROQ_MODEL")
groq_base_url = os.getenv("GROQ_BASE_URL")

if groq_api_key:
    print(f"Groq API Key exists: {groq_api_key[:4]}, {groq_model[-8:]}, {groq_base_url[-8:]}")
else:
    print("Groq API Key not set.")

Groq API Key exists: gsk_, -oss-20b, penai/v1

In [3]:
# How many characters in all the documents?

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)

print(f"Found {len(files)} files in the knowledge base.")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as file:
        entire_knowledge_base += file.read()
        entire_knowledge_base += "\n\n"  # Add a newline between files  

print(f"Total characters in the knowledge base: {len(entire_knowledge_base)}")

Found 76 files in the knowledge base.

Total characters in the knowledge base: 304434

In [4]:
# Calculate the number of tokens in the entire knowledge base using tiktoken

from transformers import AutoTokenizer

# This downloads the exact token layout mapping file for your specific model
tokenizer = AutoTokenizer.from_pretrained(groq_model, use_auth_token=groq_api_key)
num_tokens = len(tokenizer.encode(entire_knowledge_base))

print(f"Exact tokens in the knowledge base for {groq_model}: {num_tokens}")

Exact tokens in the knowledge base for openai/gpt-oss-20b: 63555

In [5]:
# Load in everything in the knowledgebase using LangChain's loaders44

folders = glob.glob("knowledge-base/*")

documents = []

for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    
    # reading and tagging metadata
    folder_docs = loader.load()

    print(f"Loaded {len(folder_docs)} documents from {doc_type} folder.")

    for doc in folder_docs:
        doc.metadata['doc_type'] = doc_type
        documents.append(doc)

print(f"Total documents loaded: {len(documents)}")

Loaded 4 documents from company folder.

Loaded 32 documents from contracts folder.

Loaded 32 documents from employees folder.

Loaded 8 documents from products folder.

Total documents loaded: 76

In [6]:
print(documents[1])

Document(
    metadata={'source': 'knowledge-base\\company\\careers.md', 'doc_type': 'company'},
    page_content="# Careers at Insurellm\n\n## Why Join Insurellm?\n\nAt Insurellm, we're not just building 
software—we're revolutionizing an entire industry. Since our founding in 2015, we've evolved from a high-growth 
startup to a lean, profitable company with 32 highly talented employees managing 32 active contracts across all 
eight of our product lines.\n\nAfter reaching 200 employees in 2020, we strategically restructured in 2022-2023 to 
focus on sustainable growth, operational excellence, and building a world-class remote-first culture. Today, we're 
a tight-knit team of exceptional professionals who deliver outsized impact through automation, AI, and strategic 
focus on high-value enterprise clients—from regional insurers to global reinsurance partners.\n\n### Our 
Culture\n\nWe live by our core values every day:\n- **Innovation First**: We encourage experimentation and creative
problem-solving\n- **Customer Obsession**: Your work directly impacts 32 active client operations spanning the 
entire insurance value chain\n- **Integrity & Transparency**: We build trust through ethical behavior and open 
communication\n- **Collaborative Excellence**: Diverse perspectives and teamwork drive our success\n\n### What We 
Offer\n\n- Competitive compensation with equity participation\n- Comprehensive health, dental, and vision 
insurance\n- Flexible working arrangements and generous PTO\n- Professional development programs and mentorship\n- 
Clear career progression paths\n- Latest technologies and tools\n- Inclusive, diverse work environment\n\n## 
Current Opportunities\n\n### Engineering\n\n**Senior Full Stack Engineer** - San Francisco, CA\n- Lead development 
of next-generation insurance platform features\n- Work with React, Node.js, Python, and cloud technologies\n- 
Mentor junior engineers and drive technical decisions\n- 5+ years experience required\n\n**Backend Software 
Engineer** - San Francisco, CA / Austin, TX\n- Build scalable microservices and APIs\n- Optimize system performance
and reliability\n- Collaborate with product and design teams\n- 3+ years experience with Java, Python, or 
Go\n\n**Frontend Developer** - Remote\n- Create intuitive user interfaces for our insurance platforms\n- Work with 
modern frameworks (React, Vue, or Angular)\n- Ensure accessibility and responsive design\n- 2+ years experience 
required\n\n**DevOps Engineer** - New York, NY\n- Manage cloud infrastructure (AWS/GCP)\n- Implement CI/CD 
pipelines and automation\n- Monitor system performance and security\n- 3+ years experience in 
DevOps/SRE\n\n**Mobile Developer (iOS/Android)** - San Francisco, CA\n- Build native mobile applications for our 
marketplace\n- Create seamless user experiences\n- Integrate with backend APIs\n- 3+ years mobile development 
experience\n\n### Data & Analytics\n\n**Senior Data Scientist** - San Francisco, CA / New York, NY\n- Develop 
predictive models for insurance risk assessment\n- Build recommendation systems for our marketplace\n- Lead 
data-driven product initiatives\n- PhD or MS in related field preferred, 4+ years experience\n\n**Data Engineer** -
Austin, TX\n- Design and maintain data pipelines\n- Build data warehousing solutions\n- Optimize data 
infrastructure for scale\n- 3+ years experience with SQL, Python, and ETL tools\n\n**Business Intelligence 
Analyst** - Chicago, IL\n- Create dashboards and reports for stakeholders\n- Analyze business metrics and trends\n-
Support data-driven decision making\n- 2+ years experience with BI tools (Tableau, Looker, etc.)\n\n### Product & 
Design\n\n**Product Manager** - San Francisco, CA\n- Define product roadmap and strategy\n- Work closely with 
engineering and design teams\n- Gather customer insights and market research\n- 3+ years product management 
experience in B2B SaaS\n\n**UX/UI Designer** - Remote\n- Design user experiences for our insurance platforms\n- 
C

In [7]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")
print(f"Example chunk content:\n{chunks[0]}")

Total chunks created: 413

Example chunk content:
page_content='# About Insurellm

Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in 
need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance 
providers.

The company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm 
(auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, 
Insurellm had reached a peak of 200 employees with 12 offices across the US.' metadata={'source': 
'knowledge-base\\company\\about.md', 'doc_type': 'company'}

In [8]:
print(chunks[100])

Document(
    metadata={
        'source': 'knowledge-base\\contracts\\Contract with GlobalRe Partners for Rellm.md',
        'doc_type': 'contracts'
    },
    page_content='13. **Climate Risk Analytics:** Forward-looking climate modeling:\n    - IPCC climate scenario 
analysis (RCP 2.6, 4.5, 8.5)\n    - Transition risk assessment\n    - Physical risk modeling for perils (hurricane,
wildfire, flood, drought)\n    - Sea level rise impact analysis\n    - Temperature trend incorporation\n    - 
Climate-adjusted pricing recommendations\n    - Stranded asset identification\n    - Green reinsurance 
opportunities\n\n---\n\n## Support\n\nInsurellm commits to comprehensive Enterprise-level support for GlobalRe 
Partners:\n\n1. **Dedicated Success Team:**\n   - Executive sponsor (CEO-level) with quarterly strategic reviews\n 
- Dedicated Senior Vice President of Customer Success with bi-weekly engagement\n   - Technical Account Manager for
platform optimization\n   - Solutions Architect team (2 FTE) for strategic initiatives\n   - Catastrophe modeling 
specialist for analytics support\n   - Quarterly executive business reviews with C-suite participation from both 
organizations'
)

#### PART B: MAKE VECTORS AND STORE IN CHROMA

In [9]:
# Embedding the chunks using Groq's embeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2") # It converts each text chunk into a 384-dimensional coordinate space.

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)

print(f"Vector store created with: {vectorstore._collection.count()} documents.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store created with: 413 documents.

In [10]:
collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)

print(f"Vector store contains {count} embeddings, each of dimension {dimensions}.")

Vector store contains 413 embeddings, each of dimension 384.

#### PART C: VISUALIZE

In [11]:
# Prework done!

result = collection.get(include=["embeddings", "metadatas", "documents"])
vectors = np.array(result["embeddings"])
metadatas = result["metadatas"]
documents = result["documents"]

doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [12]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-Distributed Stochastic Neighbor Embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create a 2D scatter plot using Plotly
fig = go.Figure(data=go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(color=colors, size=10, opacity=0.7),
    text=[f"{metadata['doc_type']}: {doc[:50]}..." for metadata, doc in zip(metadatas, documents)],
    hoverinfo='text'
))

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [13]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()